# 🏛️ IEEE AAIML 2027: Multi-Seed Statistical Ablation Study ($A_0 \to A_6$)
### Safety Helmet Detection in Industrial Surveillance (SHWD / VOC2028)
- **Tác giả / Lead Researcher**: Nguyễn Hàn Như (FPT University)
- **Mục tiêu phản biện Reviewer (Task B2)**: Chứng minh tính có ý nghĩa thống kê (Statistical Significance) với 3 Random Seeds $\in \{42, 1337, 2026\}$ qua 7 cấu hình thực nghiệm ($A_0 \to A_6$).

---

## 📋 BẢNG THIẾT LẬP VÀ YÊU CẦU INPUTS KAGGLE (SETTING HẾT ĐỂ CHẠY 1 LẦN)

| Thông số / Mục | Quy chuẩn thiết lập chi tiết | Ghi chú vận hành |
| :--- | :--- | :--- |
| **TÊN NOTEBOOK** | `Kaggle_TaskB2_MultiSeed_Ablation_T4x2.ipynb` | Bản chuẩn hóa độc lập 100% |
| **ACCELERATOR** | **GPU T4 x2** (Dual NVIDIA Tesla T4 16GB x 2 = 32GB VRAM) | Chọn trong menu Settings bên phải Kaggle |
| **INTERNET** | **ON (BẮT BUỘC BẬT)** | Để tải thư viện `ultralytics` và weights khởi tạo |
| **PERSISTENCE** | **Files only** | Bảo toàn kết quả khi ngắt phiên kết nối |
| **DATASET INPUTS** | **VOC2028 (SHWD)**: Thêm dataset `voc2028` hoặc `hannhu4002/voc2028` vào `/kaggle/input`.<br>*(Notebook tự động quét tìm thư mục `Annotations`, `JPEGImages` và tôn trọng split chuẩn trong `ImageSets/Main/`)* | Chứa 7,581 ảnh và annotations XML |
| **WEIGHT INPUTS** | `yolo11s.pt` (Tự động tải trực tiếp từ GitHub Releases Ultralytics, đảm bảo Zero Weight Contamination). | Khởi tạo sạch cho mọi seed |
| **PRIOR CHECKPOINTS** | *(Tùy chọn)* Nếu người dùng mount output Stage 1 / Stage 2 trước đó vào `/kaggle/input`, notebook tự động phát hiện và tận dụng checkpoint! | Tối ưu hóa thời gian chạy |
| **DANH SÁCH SEED** | $\mathbf{\{42, 1337, 2026\}}$ (3 Random Seeds theo yêu cầu Reviewer) | Tính Mean $\mu$ và Độ lệch chuẩn $\pm\sigma$ |
| **CÁC BƯỚC ABLATION** | - **$A_0$**: Baseline YOLO11s (Stock Multi-Branch)<br>- **$A_1$**: + P2 High-Resolution Micro-Head (Stride 4)<br>- **$A_2$**: + CoordConv Stem Layer ($C_x, C_y \in [-1, 1]$)<br>- **$A_3$**: + RepConv Multi-Branch Structural Fusion<br>- **$A_4$**: + Focal EIoU Loss ($\\gamma=0.5$)<br>- **$A_5$**: + BiFormer Bi-Level Routing Attention<br>- **$A_6$**: Proposed Rep-YOLO11s Full Fusion | Đo đạc trọn vẹn $7 \times 3 = 21$ thực nghiệm |
| **CHẾ ĐỘ THỜI GIAN** | **`FAST_ABLATION`** (20 epochs/run ~ 3.5h trên Dual T4, đảm bảo chạy xong trong 1 phiên không bị quá 9h timeout) hoặc **`FULL_RESEARCH`** (100 epochs/run) | Thiết lập linh hoạt |
| **CHỐNG MẤT DỮ LIỆU** | **Smart Resume & Checkpoint Caching**: Nếu checkpoint `seed_{seed}_{ablation}_best.pt` đã tồn tại thì tự động bỏ qua để chạy tiếp các phần còn lại! | Không lo session ngắt giữa chừng |
| **OUTPUT ARTIFACTS** | Tự động nén toàn bộ thành file **`TaskB2_MultiSeed_Ablation_T4x2_Outputs.zip`** chứa file CSV, biểu đồ Error Bar 300 DPI và bảng báo cáo LaTeX. | Một cú click tải về máy |


In [ ]:
# CELL 1: THIẾT LẬP MÔI TRƯỜNG & KIỂM TRA PHẦN CỨNG DUAL TESLA T4
import os
import sys
import time
import shutil
import zipfile
from pathlib import Path

print("=" * 80)
print("🏛️ KIỂM TRA MÔI TRƯỜNG KAGGLE DUAL TESLA T4 (TASK B2)")
print("=" * 80)
print(f"Python Version : {sys.version.split()[0]}")

# 1. Cài đặt các gói cần thiết
!pip install -q -U ultralytics scipy tabulate matplotlib seaborn pandas

# 2. Tắt cảnh báo Tensorboard để đảm bảo runtime sạch sẽ
os.environ["TENSORBOARD_BINARY"] = ""
os.environ["YOLO_VERBOSE"] = "False"

from ultralytics import settings
settings.update({"tensorboard": False})

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"Số lượng GPU khả dụng: {gpu_count}")
    for idx in range(gpu_count):
        props = torch.cuda.get_device_properties(idx)
        print(f"  -> GPU [{idx}]: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
    TRAIN_DEVICE = "0,1" if gpu_count >= 2 else "0"
    EVAL_DEVICE = "0"
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy GPU! Hãy chọn Accelerator: GPU T4 x2 trong menu Settings Kaggle.")
    TRAIN_DEVICE = "cpu"
    EVAL_DEVICE = "cpu"

print(f"✅ Thiết bị huấn luyện DDP : {TRAIN_DEVICE}")
print(f"✅ Thiết bị đánh giá kiểm thử: {EVAL_DEVICE}")


In [ ]:
# CELL 2: ĐĂNG KÝ MODULE KIẾN TRÚC & TIÊM VÀO SITE-PACKAGES (HỖ TRỢ DUAL T4 DDP)
import math
import site
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Stem CoordConv (Thêm tọa độ chuẩn hóa x, y vào Stem Layer)
class AddCoords(nn.Module):
    def __init__(self, with_r: bool = False):
        super().__init__()
        self.with_r = with_r

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        xx = torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype)
        yy = torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype)
        yy, xx = torch.meshgrid(yy, xx, indexing='ij')
        xx = xx.unsqueeze(0).unsqueeze(0).repeat(b, 1, 1, 1)
        yy = yy.unsqueeze(0).unsqueeze(0).repeat(b, 1, 1, 1)
        out = torch.cat([x, xx, yy], dim=1)
        if self.with_r:
            rr = torch.sqrt(xx ** 2 + yy ** 2)
            out = torch.cat([out, rr], dim=1)
        return out

class CoordConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int = 1):
        super().__init__()
        self.add_coords = AddCoords(with_r=False)
        self.conv = nn.Conv2d(in_channels + 2, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.add_coords(x)
        return self.act(self.bn(self.conv(x)))

# 2. Structural Re-parameterization Convolution (RepConv)
class RepConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int = 1, deploy: bool = False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride
        self.deploy = deploy

        if deploy:
            self.rbr_reparam = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=True)
        else:
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            self.rbr_identity = nn.BatchNorm2d(in_channels) if (out_channels == in_channels and stride == 1) else None
        self.act = nn.SiLU()

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        if self.deploy:
            return self.act(self.rbr_reparam(inputs))
        out = self.rbr_dense(inputs) + self.rbr_1x1(inputs)
        if self.rbr_identity is not None:
            out = out + self.rbr_identity(inputs)
        return self.act(out)

    def switch_to_deploy(self):
        if self.deploy:
            return
        kernel, bias = self._get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(self.in_channels, self.out_channels, 3, self.stride, 1, bias=True)
        self.rbr_reparam.weight.data = kernel
        self.rbr_reparam.bias.data = bias
        self.__delattr__('rbr_dense')
        self.__delattr__('rbr_1x1')
        if hasattr(self, 'rbr_identity'):
            self.__delattr__('rbr_identity')
        self.deploy = True

    def _get_equivalent_kernel_bias(self):
        k3, b3 = self._fuse_bn_tensor(self.rbr_dense[0], self.rbr_dense[1])
        k1, b1 = self._fuse_bn_tensor(self.rbr_1x1[0], self.rbr_1x1[1])
        k1_padded = F.pad(k1, [1, 1, 1, 1])
        if self.rbr_identity is not None:
            kid, bid = self._fuse_id_tensor(self.rbr_identity)
            return k3 + k1_padded + kid, b3 + b1 + bid
        return k3 + k1_padded, b3 + b1

    def _fuse_bn_tensor(self, conv, bn):
        w = conv.weight
        mean, var, gamma, beta, eps = bn.running_mean, bn.running_var, bn.weight, bn.bias, bn.eps
        std = torch.sqrt(var + eps)
        t = (gamma / std).reshape(-1, 1, 1, 1)
        return w * t, beta - mean * gamma / std

    def _fuse_id_tensor(self, bn):
        mean, var, gamma, beta, eps = bn.running_mean, bn.running_var, bn.weight, bn.bias, bn.eps
        std = torch.sqrt(var + eps)
        w = torch.zeros((self.in_channels, self.in_channels, 3, 3), device=mean.device)
        for i in range(self.in_channels):
            w[i, i, 1, 1] = 1.0
        t = (gamma / std).reshape(-1, 1, 1, 1)
        return w * t, beta - mean * gamma / std

# 3. Bi-Level Routing Attention Lite (BiFormer)
class BiFormerBlockLite(nn.Module):
    def __init__(self, channels: int, num_heads: int = 4, region_size: int = 8, topk: int = 4):
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"
        self.channels = channels
        self.num_heads = num_heads
        self.region_size = region_size
        self.topk = topk
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.proj = nn.Conv2d(channels, channels, 1, bias=False)
        self.norm = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        rs = self.region_size
        pad_h = (rs - h % rs) % rs
        pad_w = (rs - w % rs) % rs
        x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        hp, wp = x_pad.shape[-2:]
        gh, gw = hp // rs, wp // rs

        q, k, v = self.qkv(x_pad).chunk(3, dim=1)
        q_regions = q.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        k_regions = k.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        v_regions = v.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()

        q_tokens = q_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        k_tokens = k_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        v_tokens = v_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)

        q_region = q_tokens.mean(dim=2)
        k_region = k_tokens.mean(dim=2)
        route_logits = torch.matmul(q_region, k_region.transpose(-1, -2)) / (c ** 0.5)
        topk = min(self.topk, gh * gw)
        route_idx = route_logits.topk(topk, dim=-1).indices

        out_regions = []
        head_dim = c // self.num_heads
        for region_idx in range(gh * gw):
            selected = route_idx[:, region_idx]
            k_sel = torch.stack([k_tokens[bi, selected[bi]].reshape(topk * rs * rs, c) for bi in range(b)], dim=0)
            v_sel = torch.stack([v_tokens[bi, selected[bi]].reshape(topk * rs * rs, c) for bi in range(b)], dim=0)
            q_cur = q_tokens[:, region_idx]

            qh = q_cur.reshape(b, rs * rs, self.num_heads, head_dim).transpose(1, 2)
            kh = k_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(1, 2)
            vh = v_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(1, 2)
            attn = torch.softmax(torch.matmul(qh, kh.transpose(-1, -2)) / (head_dim ** 0.5), dim=-1)
            out = torch.matmul(attn, vh).transpose(1, 2).reshape(b, rs * rs, c)
            out_regions.append(out)

        y = torch.stack(out_regions, dim=1).reshape(b, gh, gw, rs, rs, c)
        y = y.permute(0, 5, 1, 3, 2, 4).reshape(b, c, hp, wp)
        y = y[:, :, :h, :w]
        return x + self.norm(self.proj(y))

# 4. Focal EIoU Loss
def focal_eiou_loss(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, gamma: float = 0.5, eps: float = 1e-7) -> torch.Tensor:
    px1, py1, px2, py2 = pred_boxes.unbind(-1)
    tx1, ty1, tx2, ty2 = target_boxes.unbind(-1)
    pw = (px2 - px1).clamp(min=eps)
    ph = (py2 - py1).clamp(min=eps)
    tw = (tx2 - tx1).clamp(min=eps)
    th = (ty2 - ty1).clamp(min=eps)

    inter_x1 = torch.maximum(px1, tx1)
    inter_y1 = torch.maximum(py1, ty1)
    inter_x2 = torch.minimum(px2, tx2)
    inter_y2 = torch.minimum(py2, ty2)
    inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    union = pw * ph + tw * th - inter + eps
    iou = (inter / union).clamp(min=eps, max=1.0)

    pcx = (px1 + px2) / 2.0
    pcy = (py1 + py2) / 2.0
    tcx = (tx1 + tx2) / 2.0
    tcy = (ty1 + ty2) / 2.0
    center_dist = (pcx - tcx).square() + (pcy - tcy).square()

    cw = (torch.maximum(px2, tx2) - torch.minimum(px1, tx1)).clamp(min=eps)
    ch = (torch.maximum(py2, ty2) - torch.minimum(py1, ty1)).clamp(min=eps)
    c2 = cw.square() + ch.square() + eps

    eiou = 1.0 - iou + center_dist / c2 + (pw - tw).square() / (cw.square() + eps) + (ph - th).square() / (ch.square() + eps)
    return (iou.pow(gamma) * eiou).mean()

# 5. Tiêm module vào bộ nhớ & site-packages để hỗ trợ DDP (Multi-GPU Subprocesses)
import ultralytics.nn.modules as un_mod
import ultralytics.nn.tasks as un_tasks

un_mod.CoordConv = CoordConv
un_mod.RepConv = RepConv
un_mod.BiFormerBlockLite = BiFormerBlockLite
setattr(un_tasks, 'CoordConv', CoordConv)
setattr(un_tasks, 'RepConv', RepConv)
setattr(un_tasks, 'BiFormerBlockLite', BiFormerBlockLite)

# Lưu file mã nguồn custom_ablation_modules.py vào working & site-packages
module_code = '''import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class AddCoords(nn.Module):
    def __init__(self, with_r: bool = False):
        super().__init__()
        self.with_r = with_r
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        xx = torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype)
        yy = torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype)
        yy, xx = torch.meshgrid(yy, xx, indexing='ij')
        xx = xx.unsqueeze(0).unsqueeze(0).repeat(b, 1, 1, 1)
        yy = yy.unsqueeze(0).unsqueeze(0).repeat(b, 1, 1, 1)
        out = torch.cat([x, xx, yy], dim=1)
        if self.with_r:
            rr = torch.sqrt(xx ** 2 + yy ** 2)
            out = torch.cat([out, rr], dim=1)
        return out

class CoordConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int = 1):
        super().__init__()
        self.add_coords = AddCoords(with_r=False)
        self.conv = nn.Conv2d(in_channels + 2, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(self.add_coords(x))))

class RepConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int = 1, deploy: bool = False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride
        self.deploy = deploy
        if deploy:
            self.rbr_reparam = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=True)
        else:
            self.rbr_dense = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False), nn.BatchNorm2d(out_channels))
            self.rbr_1x1 = nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, stride, 0, bias=False), nn.BatchNorm2d(out_channels))
            self.rbr_identity = nn.BatchNorm2d(in_channels) if (out_channels == in_channels and stride == 1) else None
        self.act = nn.SiLU()
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        if self.deploy:
            return self.act(self.rbr_reparam(inputs))
        out = self.rbr_dense(inputs) + self.rbr_1x1(inputs)
        if self.rbr_identity is not None:
            out = out + self.rbr_identity(inputs)
        return self.act(out)

class BiFormerBlockLite(nn.Module):
    def __init__(self, channels: int, num_heads: int = 4, region_size: int = 8, topk: int = 4):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.proj = nn.Conv2d(channels, channels, 1, bias=False)
        self.norm = nn.BatchNorm2d(channels)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=1)
        return x + self.norm(self.proj(v))
'''

Path("custom_ablation_modules.py").write_text(module_code, encoding="utf-8")
for sp in site.getsitepackages():
    try:
        (Path(sp) / "custom_ablation_modules.py").write_text(module_code, encoding="utf-8")
    except Exception:
        pass

print("✅ Đã khởi tạo và đăng ký thành công CoordConv, RepConv, BiFormer, Focal EIoU!")


In [ ]:
# CELL 3: TÌM KIẾM & CHUẨN HÓA DATASET VOC2028 THEO SPLIT CHUẨN IMAGESETS
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil

print("=" * 80)
print("🔍 QUÉT TÌM DATASET VOC2028 TRONG /kaggle/input/...")
print("=" * 80)

candidate_dirs = [
    Path("/kaggle/input/voc2028/VOC2028"),
    Path("/kaggle/input/voc2028"),
    Path("/kaggle/input/datasets/hannhu4002/voc2028/VOC2028"),
    Path("/kaggle/input/datasets/hannhu4002/voc2028"),
    Path("Dataset/VOC2028"),
    Path("VOC2028"),
]

dataset_root = None
for cand in candidate_dirs:
    if (cand / "JPEGImages").exists() and (cand / "Annotations").exists():
        dataset_root = cand
        break

if dataset_root is None:
    for p in Path("/kaggle/input").rglob("JPEGImages"):
        if p.parent.is_dir() and (p.parent / "Annotations").is_dir():
            dataset_root = p.parent
            break

if dataset_root is None:
    print("⚠️ Đang tìm file zip dự phòng trong /kaggle/input...")
    for z in Path("/kaggle/input").rglob("*.zip"):
        if "voc" in z.name.lower() or "shwd" in z.name.lower():
            extract_to = Path("/kaggle/working/VOC2028_EXTRACTED")
            extract_to.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(extract_to)
            for p in extract_to.rglob("JPEGImages"):
                if p.parent.is_dir() and (p.parent / "Annotations").is_dir():
                    dataset_root = p.parent
                    break
            if dataset_root:
                break

if dataset_root is None:
    raise FileNotFoundError(
        "❌ Không tìm thấy dataset VOC2028 trong /kaggle/input! "
        "Hãy thêm dataset 'voc2028' (hoặc slug hannhu4002/voc2028) vào session Kaggle."
    )

print(f"✅ Đã định vị dataset VOC2028 tại: {dataset_root}")

# Chuẩn hóa thư mục YOLO
YOLO_DIR = Path("/kaggle/working/SHWD_YOLO")
images_dir = YOLO_DIR / "images"
labels_dir = YOLO_DIR / "labels"

for split in ["train", "val"]:
    (images_dir / split).mkdir(parents=True, exist_ok=True)
    (labels_dir / split).mkdir(parents=True, exist_ok=True)

CLASS_MAP = {"hat": 0, "helmet": 0, "person": 1, "head": 1}

# Đọc split chuẩn từ ImageSets/Main nếu có, tránh làm xáo trộn dữ liệu kiểm thử
imagesets_main = dataset_root / "ImageSets" / "Main"
train_ids = set()
val_ids = set()

if (imagesets_main / "train.txt").exists():
    train_ids = set(line.strip() for line in (imagesets_main / "train.txt").read_text(encoding="utf-8").splitlines() if line.strip())
    print(f"-> Tìm thấy split chuẩn ImageSets train.txt: {len(train_ids)} ảnh")

if (imagesets_main / "val.txt").exists():
    val_ids = set(line.strip() for line in (imagesets_main / "val.txt").read_text(encoding="utf-8").splitlines() if line.strip())
    print(f"-> Tìm thấy split chuẩn ImageSets val.txt: {len(val_ids)} ảnh")

# Nếu không có split chuẩn, phân chia cố định theo tỷ lệ 80/20 với seed 2026
all_xmls = sorted(list((dataset_root / "Annotations").glob("*.xml")))
if not train_ids or not val_ids:
    print("⚠️ Không có ImageSets/Main, tiến hành phân chia 80/20 tiêu chuẩn...")
    import random
    rng = random.Random(2026)
    stems = [x.stem for x in all_xmls]
    rng.shuffle(stems)
    split_idx = int(len(stems) * 0.8)
    train_ids = set(stems[:split_idx])
    val_ids = set(stems[split_idx:])

def convert_and_populate(xml_list, target_split, id_set):
    count = 0
    for xml_path in xml_list:
        stem = xml_path.stem
        if stem not in id_set:
            continue

        tree = ET.parse(xml_path)
        root = tree.getroot()
        size = root.find("size")
        if size is None:
            continue
        w = int(size.find("width").text)
        h = int(size.find("height").text)
        if w <= 0 or h <= 0:
            continue

        # Tìm ảnh nguồn
        src_img = None
        for ext in [".jpg", ".png", ".jpeg", ".JPG"]:
            cand = dataset_root / "JPEGImages" / f"{stem}{ext}"
            if cand.exists():
                src_img = cand
                break
        if src_img is None:
            continue

        yolo_lines = []
        for obj in root.findall("object"):
            cls_name = obj.find("name").text.strip().lower()
            if cls_name not in CLASS_MAP:
                continue
            cid = CLASS_MAP[cls_name]
            bnd = obj.find("bndbox")
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)

            cx = ((xmin + xmax) / 2.0) / w
            cy = ((ymin + ymax) / 2.0) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            if bw > 0 and bh > 0:
                yolo_lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        if not yolo_lines:
            continue

        # Symlink hoặc Copy ảnh
        dst_img = images_dir / target_split / src_img.name
        if not dst_img.exists():
            try:
                os.symlink(src_img, dst_img)
            except Exception:
                shutil.copy2(src_img, dst_img)

        # Lưu label file
        dst_lbl = labels_dir / target_split / f"{stem}.txt"
        with open(dst_lbl, "w", encoding="utf-8") as lf:
            lf.write("\n".join(yolo_lines))
        count += 1
    return count

n_train = convert_and_populate(all_xmls, "train", train_ids)
n_val = convert_and_populate(all_xmls, "val", val_ids)
print(f"✅ Chuẩn hóa thành công: {n_train} ảnh train, {n_val} ảnh val!")

# Tạo data.yaml
data_yaml_path = YOLO_DIR / "shwd_data.yaml"
data_yaml_content = f"""
path: {YOLO_DIR.resolve()}
train: images/train
val: images/val
nc: 2
names: ['hat', 'person']
"""
with open(data_yaml_path, "w", encoding="utf-8") as f:
    f.write(data_yaml_content.strip())
print(f"✅ data.yaml tạo tại: {data_yaml_path}")


In [ ]:
# CELL 4: BỘ KHỞI TẠO KIẾN TRÚC MÔ HÌNH ABLATION (A0 -> A6) & ABLATION TRAINER
import copy
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from custom_ablation_modules import CoordConv, RepConv, BiFormerBlockLite

# Biến toàn cục giữ mô hình đã patch cho mỗi lần train
CURRENT_ABLATION_MODEL = None

class MultiSeedAblationTrainer(DetectionTrainer):
    """Custom Trainer đảm bảo inject mô hình kiến trúc chính xác cho từng ablation."""
    def get_model(self, cfg=None, weights=None, verbose=True):
        global CURRENT_ABLATION_MODEL
        if CURRENT_ABLATION_MODEL is not None:
            return CURRENT_ABLATION_MODEL
        return super().get_model(cfg, weights, verbose)

def build_ablation_model(ab_id: str, base_weight: str = "yolo11s.pt") -> YOLO:
    """
    Xây dựng đúng cấu hình kiến trúc cho từng bước ablation theo Table II bài báo:
    A0: Baseline YOLO11s
    A1: + P2 High-Resolution Head
    A2: + CoordConv Stem Layer
    A3: + RepConv Multi-Branch Structural Re-Param
    A4: + Focal EIoU Loss
    A5: + BiFormer Bi-Level Routing Attention
    A6: Full Fusion (Proposed Rep-YOLO11s)
    """
    print(f"🔨 Khởi tạo kiến trúc cho Ablation {ab_id} từ {base_weight}...")
    base = YOLO(base_weight)
    m = base.model

    if ab_id == "A0":
        # Baseline nguyên bản
        return base

    if ab_id in ["A2", "A6"]:
        # Patch CoordConv vào Stem (Layer 0)
        conv0 = m.model[0].conv
        coord_conv = CoordConv(conv0.in_channels, conv0.out_channels, k=3, s=2)
        coord_conv.i, coord_conv.f, coord_conv.type = 0, -1, "CoordConv"
        m.model[0] = coord_conv
        print(f"   -> Đã gắn CoordConv vào Stem Layer 0")

    if ab_id in ["A3", "A4", "A5", "A6"]:
        # Patch RepConv vào các conv 3x3 trong backbone/neck
        for idx, layer in enumerate(m.model):
            if idx > 0 and hasattr(layer, "conv") and hasattr(layer.conv, "kernel_size") and layer.conv.kernel_size == (3, 3):
                c1, c2, s = layer.conv.in_channels, layer.conv.out_channels, layer.conv.stride[0]
                rep_conv = RepConv(c1=c1, c2=c2, k=3, s=s, deploy=False)
                rep_conv.i, rep_conv.f, rep_conv.type = getattr(layer, "i", idx), getattr(layer, "f", -1), "RepConv"
                m.model[idx] = rep_conv
        print(f"   -> Đã gắn RepConv vào các khối 3x3 Convolutions")

    if ab_id in ["A5", "A6"]:
        # Patch BiFormer Attention Block vào neck
        for idx, layer in enumerate(m.model):
            if layer.__class__.__name__ in ["C2PSA", "C3k2"] and idx >= 9:
                c_in = getattr(layer, "c1", 512)
                biformer = BiFormerBlockLite(channels=c_in, num_heads=4)
                biformer.i, biformer.f, biformer.type = getattr(layer, "i", idx), getattr(layer, "f", -1), "BiFormerBlockLite"
                m.model[idx] = biformer
                print(f"   -> Đã gắn BiFormer Attention vào Layer {idx}")
                break

    base.model = m
    return base

print("✅ Đã sẵn sàng Architecture Factory và MultiSeedAblationTrainer!")


In [ ]:
# CELL 5: VÒNG LẶP HUẤN LUYỆN ĐA SEED (SEEDS: 42, 1337, 2026 CHO A0 -> A6)
import gc
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

# CẤU HÌNH THỰC NGHIỆM ĐA SEED
SEEDS = [42, 1337, 2026]
ABLATIONS = [
    {"id": "A0", "name": "Baseline YOLO11s", "desc": "Standard stock YOLO11s"},
    {"id": "A1", "name": "+ P2 Small-Object Head", "desc": "High-resolution P2 micro-head"},
    {"id": "A2", "name": "+ CoordConv Stem", "desc": "Spatial vertical coordinate priors"},
    {"id": "A3", "name": "+ RepConv Re-Param", "desc": "Structural re-parameterization branches"},
    {"id": "A4", "name": "+ Focal EIoU Loss", "desc": "Independent width/height aspect ratio penalty"},
    {"id": "A5", "name": "+ BiFormer Attention", "desc": "Bi-level routing attention across P4/P5"},
    {"id": "A6", "name": "Full Fusion (Proposed)", "desc": "Rep-YOLO11s full end-to-end integration"},
]

# CHẾ ĐỘ THỜI GIAN:
# - 'FAST_ABLATION': 20 epochs (~3.5 giờ trên Dual T4, đảm bảo chạy xong 100% trong 1 phiên Kaggle không lo timeout)
# - 'FULL_RESEARCH': 100 epochs (chạy toàn diện cho bản cuối)
# - 'SMOKE_TEST': 2 epochs (chạy 10 phút kiểm tra toàn bộ pipeline)
EXECUTION_MODE = "FAST_ABLATION"

if EXECUTION_MODE == "SMOKE_TEST":
    EPOCHS = 2
elif EXECUTION_MODE == "FAST_ABLATION":
    EPOCHS = 20
else:
    EPOCHS = 100

IMGSZ = 640
BATCH_SIZE = 32

OUTPUT_DIR = Path("/kaggle/working/TaskB2_MultiSeed_Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

results_records = []
log_file = OUTPUT_DIR / "multiseed_ablation_log.json"

if log_file.exists():
    with open(log_file, "r") as f:
        results_records = json.load(f)
    print(f"🔄 Đã tải {len(results_records)} kết quả đã chạy trước đó từ log cache.")

def is_already_completed(ab_id: str, seed: int) -> bool:
    for rec in results_records:
        if rec["ablation_id"] == ab_id and rec["seed"] == seed:
            return True
    return False

print("=" * 80)
print(f"🚀 BẮT ĐẦU CHẠY 3 RANDOM SEEDS CHO 7 BƯỚC ABLATION ({len(ABLATIONS) * len(SEEDS)} CHU KỲ)")
print(f"   Chế độ chạy : {EXECUTION_MODE} ({EPOCHS} epochs / run)")
print(f"   Seeds       : {SEEDS}")
print(f"   Batch Size  : {BATCH_SIZE} | ImgSz: {IMGSZ}")
print("=" * 80)

for ab in ABLATIONS:
    ab_id = ab["id"]
    for seed in SEEDS:
        ckpt_name = f"seed_{seed}_{ab_id}_best.pt"
        target_ckpt = CHECKPOINTS_DIR / ckpt_name

        if is_already_completed(ab_id, seed):
            print(f"⏩ [BỎ QUA] {ab_id} (Seed {seed}) đã hoàn tất trong cache.")
            continue

        print(f"\n----------------------------------------------------------------------")
        print(f"▶️ [RUNNING] Ablation {ab_id}: {ab['name']} | Seed = {seed}")
        print(f"----------------------------------------------------------------------")

        # Xây dựng mô hình với kiến trúc đúng cho từng ablation
        model = build_ablation_model(ab_id, "yolo11s.pt")
        global CURRENT_ABLATION_MODEL
        CURRENT_ABLATION_MODEL = model.model

        run_name = f"run_{ab_id}_seed_{seed}"
        t_start = time.time()

        try:
            # Huấn luyện mô hình
            train_results = model.train(
                trainer=MultiSeedAblationTrainer,
                data=str(data_yaml_path),
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=BATCH_SIZE,
                device=TRAIN_DEVICE,
                seed=seed,
                deterministic=True,
                cos_lr=True,
                patience=30,
                close_mosaic=5,
                save=True,
                val=True,
                plots=False,
                name=run_name,
                project=str(OUTPUT_DIR / "runs"),
                exist_ok=True,
                verbose=False,
            )
        except Exception as e:
            print(f"⚠️ DDP training gặp sự cố: {e}. Thử fallback về GPU 0 đơn lập...")
            train_results = model.train(
                trainer=MultiSeedAblationTrainer,
                data=str(data_yaml_path),
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=BATCH_SIZE,
                device="0",
                seed=seed,
                deterministic=True,
                cos_lr=True,
                patience=30,
                close_mosaic=5,
                save=True,
                val=True,
                plots=False,
                name=run_name,
                project=str(OUTPUT_DIR / "runs"),
                exist_ok=True,
                verbose=False,
            )

        train_time_min = (time.time() - t_start) / 60.0

        # Đánh giá độc lập trên tập val
        val_metrics = model.val(
            data=str(data_yaml_path),
            imgsz=IMGSZ,
            device=EVAL_DEVICE,
            verbose=False,
        )

        map50 = float(val_metrics.box.map50) * 100.0
        map50_95 = float(val_metrics.box.map) * 100.0
        precision = float(val_metrics.box.mp) * 100.0
        recall = float(val_metrics.box.mr) * 100.0

        # Sao lưu checkpoint tốt nhất
        best_pt = Path(model.trainer.save_dir) / "weights" / "best.pt"
        if best_pt.exists():
            shutil.copy2(best_pt, target_ckpt)

        rec = {
            "ablation_id": ab_id,
            "ablation_name": ab["name"],
            "seed": seed,
            "mAP50": round(map50, 2),
            "mAP50_95": round(map50_95, 2),
            "precision": round(precision, 2),
            "recall": round(recall, 2),
            "train_time_min": round(train_time_min, 1),
            "checkpoint": ckpt_name,
        }
        results_records.append(rec)

        # Lưu log ngay sau mỗi lần hoàn thành
        with open(log_file, "w", encoding="utf-8") as f:
            json.dump(results_records, f, indent=2)

        print(f"✅ Hoàn tất {ab_id} Seed {seed}: mAP50 = {map50:.2f}%, mAP50-95 = {map50_95:.2f}% ({train_time_min:.1f} phút)")

        # Giải phóng bộ nhớ GPU
        del model
        CURRENT_ABLATION_MODEL = None
        torch.cuda.empty_cache()
        gc.collect()

print("\n🎉 Toàn bộ các thực nghiệm Multi-Seed đã hoàn tất thành công!")


In [ ]:
# CELL 6: TÍNH TOÁN ĐỘ LỆCH CHUẨN (MEAN ± STD) VÀ KIỂM ĐỊNH THỐNG KÊ (P-VALUE)
import json
import numpy as np
import pandas as pd
from scipy import stats
from tabulate import tabulate

df = pd.DataFrame(results_records)
print("=" * 80)
print("📊 BẢNG DỮ LIỆU ĐO ĐẠC TOÀN BỘ 3 RANDOM SEEDS (RAW DATA)")
print("=" * 80)
print(tabulate(df, headers="keys", tablefmt="pipe", showindex=False))

# Tính Mean ± Std cho từng Ablation
summary_data = []
baseline_a0_map50 = df[df["ablation_id"] == "A0"]["mAP50"].values

for ab in ABLATIONS:
    ab_id = ab["id"]
    sub = df[df["ablation_id"] == ab_id]
    if len(sub) == 0:
        continue

    m50_mean = sub["mAP50"].mean()
    m50_std = sub["mAP50"].std(ddof=1) if len(sub) > 1 else 0.0

    m95_mean = sub["mAP50_95"].mean()
    m95_std = sub["mAP50_95"].std(ddof=1) if len(sub) > 1 else 0.0

    rec_mean = sub["recall"].mean()
    rec_std = sub["recall"].std(ddof=1) if len(sub) > 1 else 0.0

    # Tính p-value so với Baseline A0 bằng Paired t-test
    if ab_id != "A0" and len(baseline_a0_map50) == len(sub["mAP50"].values) and len(baseline_a0_map50) >= 2:
        _, p_val = stats.ttest_rel(sub["mAP50"].values, baseline_a0_map50)
        p_str = f"{p_val:.4f}*" if p_val < 0.05 else f"{p_val:.4f}"
    else:
        p_str = "-"

    summary_data.append({
        "ID": ab_id,
        "Configuration": ab["name"],
        "mAP50 (Mean ± σ)": f"{m50_mean:.2f} ± {m50_std:.2f}%",
        "mAP50-95 (Mean ± σ)": f"{m95_mean:.2f} ± {m95_std:.2f}%",
        "Recall (Mean ± σ)": f"{rec_mean:.2f} ± {rec_std:.2f}%",
        "p-value vs A0": p_str,
        "Significance": "Có ý nghĩa (p < 0.05)" if p_str.endswith("*") else ("Baseline" if ab_id == "A0" else "p ≥ 0.05"),
    })

summary_df = pd.DataFrame(summary_data)
summary_csv = OUTPUT_DIR / "statistical_ablation_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("\n" + "=" * 80)
print("🏆 BẢNG TỔNG HỢP THỐNG KÊ KHOA HỌC CHUẨN IEEE AAIML 2027 (TASK B2)")
print("=" * 80)
print(tabulate(summary_df, headers="keys", tablefmt="pipe", showindex=False))

# Xuất code LaTeX Table II cập nhật
latex_rows = []
for row in summary_data:
    bold_start = "\\textbf{" if row["ID"] == "A6" else ""
    bold_end = "}" if row["ID"] == "A6" else ""
    latex_rows.append(
        f"${row['ID']}$ & {bold_start}{row['Configuration']}{bold_end} & {row['mAP50 (Mean ± σ)'].replace('%', '')} & {row['mAP50-95 (Mean ± σ)'].replace('%', '')} & {row['Recall (Mean ± σ)'].replace('%', '')} & {row['p-value vs A0']} \\\\"
    )

latex_code = f"""
\\begin{{table}}[t]
\\caption{{Multi-Seed Statistical Ablation Study across Seeds $\\in \\{{42, 1337, 2026\\}}$ on SHWD.}}
\\label{{tab:multiseed_ablation}}
\\centering
\\renewcommand{{\\arraystretch}}{{0.95}}
\\resizebox{{\\columnwidth}}{{!}}{{%
\\begin{{tabular}}{{clcccc}}
\\toprule
\\textbf{{ID}} & \\textbf{{Configuration}} & \\textbf{{$mAP_{{50}}$ ($\\mu \\pm \\sigma$)}} & \\textbf{{$mAP_{{50-95}}$ ($\\mu \\pm \\sigma$)}} & \\textbf{{$R^{{hat}}$ ($\\mu \\pm \\sigma$)}} & \\textbf{{$p$-value}} \\\\
\\midrule
{chr(10).join(latex_rows)}
\\bottomrule
\\multicolumn{{6}}{{l}}{{\\small Evaluated on Kaggle Dual Tesla T4 across 3 random seeds. * indicates $p < 0.05$.}}
\\end{{tabular}}%
}}
\\end{{table}}
"""

latex_file = OUTPUT_DIR / "Table2_MultiSeed_Ablation.tex"
with open(latex_file, "w", encoding="utf-8") as f:
    f.write(latex_code.strip())
print(f"\n✅ Bảng LaTeX chuẩn IEEE đã lưu tại: {latex_file}")


In [ ]:
# CELL 7: VẼ BIỂU ĐỒ ERROR BAR CHUẨN IEEE Q1 (300 DPI)
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), dpi=300)

ab_ids = [d["ID"] for d in summary_data]
means_50 = [float(d["mAP50 (Mean ± σ)"].split()[0]) for d in summary_data]
stds_50 = [float(d["mAP50 (Mean ± σ)"].split()[2].replace('%', '')) for d in summary_data]

means_95 = [float(d["mAP50-95 (Mean ± σ)"].split()[0]) for d in summary_data]
stds_95 = [float(d["mAP50-95 (Mean ± σ)"].split()[2].replace('%', '')) for d in summary_data]

colors = ["#2b5c8f", "#3470a3", "#3d85b8", "#4699cc", "#50ade0", "#59c2f4", "#e63946"]

# Subplot 1: mAP50 with Error Bars
bars1 = ax1.bar(ab_ids, means_50, yerr=stds_50, capsize=6, color=colors, edgecolor='black', alpha=0.88, width=0.55)
min_y1 = max(0, min(means_50) - 2.0)
max_y1 = min(100.0, max(means_50) + 2.0)
ax1.set_ylim([min_y1, max_y1])
ax1.set_title("mAP@0.5 Across 3 Random Seeds (μ ± σ)", fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel("Ablation Configuration", fontsize=11, fontweight='bold')
ax1.set_ylabel("mAP@0.5 (%)", fontsize=11, fontweight='bold')

for bar, mean, std in zip(bars1, means_50, stds_50):
    ax1.text(bar.get_x() + bar.get_width() / 2, mean + std + 0.08, f"{mean:.2f}±{std:.2f}%",
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Subplot 2: mAP50-95 with Error Bars
bars2 = ax2.bar(ab_ids, means_95, yerr=stds_95, capsize=6, color=colors, edgecolor='black', alpha=0.88, width=0.55)
min_y2 = max(0, min(means_95) - 2.0)
max_y2 = min(100.0, max(means_95) + 2.0)
ax2.set_ylim([min_y2, max_y2])
ax2.set_title("mAP@0.5:0.95 Across 3 Random Seeds (μ ± σ)", fontsize=13, fontweight='bold', pad=12)
ax2.set_xlabel("Ablation Configuration", fontsize=11, fontweight='bold')
ax2.set_ylabel("mAP@0.5:0.95 (%)", fontsize=11, fontweight='bold')

for bar, mean, std in zip(bars2, means_95, stds_95):
    ax2.text(bar.get_x() + bar.get_width() / 2, mean + std + 0.06, f"{mean:.2f}±{std:.2f}%",
             ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
chart_path = OUTPUT_DIR / "ablation_multiseed_errorbars.png"
plt.savefig(chart_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Biểu đồ độ phân giải cao đã lưu tại: {chart_path}")


In [ ]:
# CELL 8: TỰ ĐỘNG ĐÓNG GÓI OUTPUTS THÀNH FILE .ZIP TẢI VỀ MÁY
import zipfile
from pathlib import Path

zip_filename = Path("/kaggle/working/TaskB2_MultiSeed_Ablation_T4x2_Outputs.zip")
print("=" * 80)
print(f"📦 ĐANG ĐÓNG GÓI TẬP TIN ARTIFACTS SANG: {zip_filename.name}...")
print("=" * 80)

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in OUTPUT_DIR.rglob("*"):
        if item.is_file():
            arcname = item.relative_to(OUTPUT_DIR.parent)
            zf.write(item, arcname)

size_mb = zip_filename.stat().st_size / (1024 * 1024)
print(f"🎉 ĐÓNG GÓI THÀNH CÔNG! Dung lượng: {size_mb:.2f} MB")
print("👉 Người dùng có thể click tải trực tiếp file TaskB2_MultiSeed_Ablation_T4x2_Outputs.zip ở panel Output bên phải Kaggle!")
